# 第 31 章 · 装配同一个 Mini Codex，并核对真实运行

各章现在共用 `code/mini_codex`：Notebook 导入最终 package 的组件，CLI 和服务入口运行同一个 `MiniCodex`、`Loop`、历史与工具路径。本章把组件存在、实际接线、任务通过、外部能力可用四类证据分开检查。

默认主实验会运行真实 Docker/Linux 样本：模型输出使用 replay，文件修改、进程、隔离和固定 5 个 case 仍须实际执行。缺 Docker 时保留 blocked；不得把后续未执行检查写成通过。这里不会调用真实模型、浏览器、语音或云服务。

最终包是可运行的教学 Coding Agent。实时语音、浏览器/设备、云服务等目前只交付相应局部组件或协议，未配置真实 provider；导入成功不能证明这些外部能力已完成。

In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'code/mini_codex/__init__.py').is_file())
sys.path.insert(0, str(ROOT/'code'))
from mini_codex.learning import load_example, show_source, fresh_run_dir, run_python, run_chapter

## 1 · 检查装配来源，再构造真正的 Engine

库存检查会真实导入最终包里的每个组件，记录源码摘要和公开接口。随后构造 `MiniCodex(AgentConfig(...))`，检查它实际拥有持久历史和 ServiceHub；此时只初始化本地状态，不执行任务。

**可改参数：** 把 `context_budget` 调小，观察配置传入同一 Engine。不要把 inventory 的 passed 解释成运行验收；它只证明当前导入来源与接口存在。

In [2]:
import json
from mini_codex.engine import AgentConfig, MiniCodex
from mini_codex.services import ServiceHub
assembly = load_example('31', 'system_audit')
study_dir = fresh_run_dir('ch31-assembly')
inventory = assembly.package_inventory()
assert inventory['passed'] and len(inventory['modules']) >= 30
assert all(row['declared_public_interfaces'] for row in inventory['modules'])
assert all(Path(row['path']).is_relative_to(ROOT/'code/mini_codex') for row in inventory['modules'])
workspace = study_dir/'workspace'
workspace.mkdir()
config = AgentConfig(workspace=workspace, state_dir=study_dir/'state', mode='replay',
                     context_budget=32000, run_dir=study_dir/'runtime')
agent = MiniCodex(config)
try:
    assert isinstance(agent.services, ServiceHub)
    assert agent.history.journal is not None
    assert agent.services.doctor()['status'] == 'passed'
    capabilities = agent.capabilities()
    assert any(c['name'] == 'computer_use' and c['status'] == 'protocol_only' for c in capabilities)
finally:
    agent.close()
(study_dir/'inventory.json').write_text(json.dumps(inventory, ensure_ascii=False, indent=2))
print({'imported_components': len(inventory['modules']), 'engine': type(agent).__name__,
       'service_boundaries': capabilities[-11:]})

{'imported_components': 36, 'engine': 'MiniCodex', 'service_boundaries': [{'name': 'telemetry', 'status': 'runtime_connected', 'scope': 'model request/result -> SQLite -> Ledger'}, {'name': 'cache_statistics', 'status': 'runtime_connected', 'scope': 'observed usage only; no cache implementation'}, {'name': 'artifact_capture', 'status': 'available_local', 'scope': 'file bytes and SHA-256'}, {'name': 'evaluation', 'status': 'available_local', 'scope': 'final demo evidence gate'}, {'name': 'hosts', 'status': 'local_mechanism', 'scope': 'in-process journal; no daemon or remote transport'}, {'name': 'cloud_artifacts', 'status': 'local_mechanism', 'scope': 'Git artifact gate; cloud service not configured'}, {'name': 'media', 'status': 'local_mechanism', 'scope': 'attachment bytes and jobs; image provider not configured'}, {'name': 'realtime', 'status': 'protocol_only', 'scope': 'event/queue logic; audio devices and service not configured'}, {'name': 'search', 'status': 'protocol_only', 'scop

## 2 · 从最终入口执行同一个完整任务

`run_chapter('31', 'replay')` 调用最终包的 demo 入口。该入口构造同一 `AgentConfig` 和 `MiniCodex`，先确认固定缺陷确实为红，再让 Loop 读取、执行、修改、复测，最后冻结候选。不是从本章另建一个简化 Loop，也不是导入旧章节的副本。

replay 只控制模型输出。真实模型能力、费用和服务延迟仍没有证据。下面默认真正运行；如果返回 blocked，保留原因，补齐环境后重跑本节。

In [3]:
demo = run_chapter('31', 'replay', env_file=None)
artifacts = None
if demo.get('status') == 'blocked':
    print({'status': 'blocked', 'reason': demo.get('reason'), 'evidence': demo.get('evidence')})
else:
    assert demo['returncode'] == 0, demo
    assert demo.get('cleanup_status') == 'confirmed', demo
    artifacts = Path(demo['artifacts'])
    assert artifacts.is_dir()
    result = json.loads((artifacts/'result.json').read_text())
    assert result['task_status'] == 'passed' and result['mode'] == 'replay'
    assert result['cleanup_status'] == 'confirmed'
    print({'task_status': result['task_status'], 'steps': result['steps'], 'artifacts': str(artifacts)})

Running checkpoint 31/replay; logs: <project>/runs/notebooks/checkpoint-31-replay-20261113T124126Z-5c5308c7


{'task_status': 'passed', 'steps': 6, 'artifacts': '<project>/runs/agent-replay-e3b27ec71a/artifacts'}


## 3 · 对冻结快照重算摘要，并核对 5 个固定 case

模型说完成只触发验证。这里从最终包读取受保护 case 合同，再用 gate 核对实际执行记录、实际值、清理状态、快照文件摘要。gate 重算字节证据，不再次运行候选；真实候选执行发生在上一节的保护验证器里。

**可改实验：** 后面的反例会删证据或修改副本，观察同一个 gate 如何拒绝。不要修改原运行记录。

In [4]:
accepted = None
if artifacts is not None:
    expected_cases = json.loads((ROOT/'code/mini_codex/assets/checks/cases.json').read_text())
    assert len(expected_cases) == 5
    accepted = assembly.evaluate_run(artifacts, expected_cases)
    assert accepted['status'] == 'passed', accepted
    final = json.loads((artifacts/'final-verification.json').read_text())
    assert final['case_ids'] == [case['id'] for case in expected_cases]
    assert final['executed_count'] == final['assertion_count'] == 5
    assert final['protection_intact'] and final['snapshot_unchanged']
    frozen_file = assembly.capture_artifact(artifacts/'snapshot/task_list.py')
    assert frozen_file['sha256'] == final['candidate']['files']['task_list.py']
    (study_dir/'accepted-demo.json').write_text(json.dumps(accepted, ensure_ascii=False, indent=2))
    print({'scope': accepted['scope'], 'verified_cases': accepted['verified_case_ids'], 'snapshot': frozen_file})
else:
    print('主任务被环境阻断：本节未取得最终快照或 case 通过证据。')

{'scope': 'final_demo', 'verified_cases': ['status-done', 'status-todo', 'all-control', 'missing-control', 'empty-control'], 'snapshot': {'path': '<project>/runs/agent-replay-e3b27ec71a/artifacts/snapshot/task_list.py', 'size': 393, 'sha256': 'ab4011f7b5b136c77a80703cef61dbeb0f51ab58ec3cdd68136d433af60df382'}}


## 4 · 历史与用量来自实际回调，不从成功结果补造

Loop 每次模型结果都会调用 `ServiceHub.on_model`。SQLite 保存 raw usage 及其存在状态，JSONL 是可重建投影；第 27 章 Ledger 与第 28 章统计函数计算同一批观察。

replay 没有 provider usage，因此本实验应显示未知用量、零覆盖，缓存比例为 None。没有 provider response ID 时保留本地 sampling_id，绝不把 HTTP request ID 或 replay 序号伪造成 response ID。

In [5]:
if artifacts is not None:
    history = json.loads((artifacts/'history.json').read_text())
    services = json.loads((artifacts/'services.json').read_text())
    run_events = [event for event in history if event['run_id'] == result['run_id']]
    model_events = [event for event in run_events if event['kind'] == 'model_result']
    assert any(event['kind'] == 'tool_result' for event in run_events)
    assert any(event['kind'] == 'final_verification' for event in run_events)
    assert any(event['kind'] == 'scheduler_terminal' for event in run_events)
    replay_scope = services['scopes']['replay']
    assert replay_scope['sampling_events'] == len(model_events) > 0
    assert 'live' not in services['scopes']
    assert replay_scope['cache']['eligible_attempts'] == 0
    assert replay_scope['cache']['cached_token_fraction'] is None
    assert replay_scope['cache']['usage_coverage'] == 0
    ledger_files = list(Path(demo['evidence']).rglob('model-usage.jsonl'))
    assert len(ledger_files) == 1
    raw_usage = [json.loads(line) for line in ledger_files[0].read_text().splitlines()]
    assert len(raw_usage) == replay_scope['sampling_events']
    assert all(row['response_id'] is None and row['identity_basis'] == 'local_sampling_id' for row in raw_usage)
    assert all(row['raw_usage'] is None for row in raw_usage)
    print({'committed_history_events': len(run_events), 'sampling_events': len(model_events),
           'usage_presence': replay_scope['usage_presence'], 'cache': replay_scope['cache'],
           'billing_claim': services['billing_claim']})
else:
    print('主任务被环境阻断：不生成替身 history 或 usage 结果。')

{'committed_history_events': 53, 'sampling_events': 6, 'usage_presence': {'not_observable': 6}, 'cache': {'evidence_kind': 'replay', 'attempts': 6, 'duplicate_records': 0, 'phase_counts': {'measure': 6}, 'status_counts': {'completed': 6}, 'eligible_attempts': 0, 'usage_coverage': 0.0, 'eligible_input_tokens': 0, 'eligible_cached_tokens': 0, 'cached_token_fraction': None, 'positive_cached_attempt_fraction': None, 'excluded': {'["[\\"replay\\",\\"replay\\"]","attempt","[\\"run-b8aed106db7c4303ad1b449d5d24d6f7\\",\\"run-b8aed106db7c4303ad1b449d5d24d6f7:turn1\\",\\"run-b8aed106db7c4303ad1b449d5d24d6f7:sample1\\"]"]': 'unverified_provider_semantics', '["[\\"replay\\",\\"replay\\"]","attempt","[\\"run-b8aed106db7c4303ad1b449d5d24d6f7\\",\\"run-b8aed106db7c4303ad1b449d5d24d6f7:turn1\\",\\"run-b8aed106db7c4303ad1b449d5d24d6f7:sample2\\"]"]': 'unverified_provider_semantics', '["[\\"replay\\",\\"replay\\"]","attempt","[\\"run-b8aed106db7c4303ad1b449d5d24d6f7\\",\\"run-b8aed106db7c4303ad1b449d5d2

## 5 · 缺证据和改快照必须拒绝 pass

先验证空证据目录确实被拒绝。若主任务已完成，再把其证据复制到实验副本，篡改冻结代码；原先所有 case 的 passed 字段即使还在，最终 gate 也必须发现摘要不一致。

这个反例核对证据绑定，不声称防住拥有可信控制器权限的攻击者。可信验证器、合同和原始证据必须位于 Agent 可写工作区之外。

In [6]:
import shutil
missing_dir = fresh_run_dir('ch31-missing-evidence')
missing = assembly.evaluate_run(missing_dir)
assert missing['status'] == 'rejected'
assert 'missing_evidence:verification' in missing['reasons']
print({'missing_evidence_gate': missing})
if artifacts is not None:
    tampered = fresh_run_dir('ch31-tampered-copy')
    for filename in ('result.json', 'final-verification.json'):
        shutil.copy2(artifacts/filename, tampered/filename)
    shutil.copytree(artifacts/'snapshot', tampered/'snapshot')
    with (tampered/'snapshot/task_list.py').open('a') as stream:
        stream.write('\n# controlled change after verification\n')
    refused = assembly.evaluate_run(tampered)
    assert refused['status'] == 'rejected'
    assert 'frozen_snapshot_hash_mismatch' in refused['reasons']
    assert assembly.evaluate_run(artifacts)['status'] == 'passed'
    print({'changed_copy': refused['reasons'], 'original_evidence': 'still_matches'})

{'missing_evidence_gate': {'status': 'rejected', 'scope': 'final_demo', 'reasons': ['missing_evidence:result', 'missing_evidence:verification'], 'artifacts': []}}
{'changed_copy': ['frozen_snapshot_hash_mismatch'], 'original_evidence': 'still_matches'}


## 源码对照与交付边界

- [受控模型 SSE 支持](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/core/tests/common/responses.rs#L1205-L1211)：控制模型响应，仍需观察实际机制。
- [补丁集成测试](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/core/tests/suite/apply_patch_cli.rs#L440-L476)：任务经过真实调用链后检查文件结果；读到测试不等于本机运行过 Codex 测试。
- [分平台 CI 配置](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/.github/workflows/rust-ci-full-nextest-platform.yml#L300-L308)：平台、构建与运行范围是证据身份的一部分。

本章的 Python package、ServiceHub 和最终 gate 是教学选择。主实验若通过，证明的是当前最终包在本次 Linux/replay 配置下完成固定任务，且持久历史、用量回调、冻结快照与固定断言一致。没有把模型回放记为真实服务成功，也没有把语音、浏览器、云端等未配置能力核销为可用。

后续扩展继续接回同一 Engine：提供真实适配器、选择明确权限、复用同一事件和工具路径，再补相应服务与取消/恢复证据。